# Notebook 45 — Option 4: direct discriminative classifier (bypass the SBI posterior)

## Purpose

One of three notebooks (44/45/46) built 2026-09-06 to test remaining options for
salvaging System II fault classification. **You are running this notebook, not
Claude** — read the markdown cells, they explain what to watch for.

## The idea

Every classification result so far (0.07 lognormal, 0.11 alpha-mixed, 0.694 historical
full-uniform) has gone through the same pipeline: train a full 5-D SBI posterior
`p(theta | x)`, sample it, then apply a hard posterior-mass threshold rule
(`sample_fault_unit`) to turn samples into a class prediction. **But classification only
needs discriminative power, not a calibrated posterior distribution.** Every failure mode
diagnosed this project (SBC miscalibration, prior-coverage mismatch, `alpha`'s seed-
dependent shrinkage) is a failure of the *posterior estimation* step — none of it is
necessarily a failure of the underlying *information content* in the 74-D summary vector
for distinguishing fault classes.

This notebook sidesteps the whole posterior-calibration question: train a direct
supervised classifier `summary -> fault_class` on simulated (summary, label) pairs, where
the label comes straight from the *true* theta used to simulate (`sample_fault_unit(theta)`),
not from posterior samples. No SBC, no calibration, no threshold-on-samples logic — just
a standard classification problem. If this scores well, it tells us the *information* for
classification is there in the summary vector even where posterior estimation currently
fails, which would argue for using/reporting a direct classifier (alongside, or instead
of, posterior-based classification) in the manuscript. If it *also* scores poorly, that
would suggest the summary vector itself doesn't cleanly separate classes at the noise
level tested, which is a more fundamental problem than anything diagnosed so far.

**Important caveat this notebook does NOT resolve**: training data still has to come from
*somewhere*, and it needs to cover the fault-relevant region of parameter space to be
useful (the exact same coverage problem motivating Options 1-3). This notebook uses
`mixed_wide_prior` (this session's best-available wide-coverage prior, already smoke-
tested) to generate training draws — if this notebook's classifier still performs badly,
first check whether it's a genuine information-content problem or just this session's
same recurring coverage-mismatch problem showing up again in a new guise.


In [ ]:
import pathlib, sys, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

from cstr_sbi.recycle.priors import mixed_wide_prior
from cstr_sbi.recycle.physics import PARAM_NAMES
from cstr_sbi.recycle.scenarios import list_closed_loop, FAULT_UNITS
from cstr_sbi.recycle.simulator import nominal_warm_start, deterministic_window, noisy_replicates
from cstr_sbi.recycle.summaries import compute_summaries, N_SUMMARIES_SC2
from sbi_pipeline import _structure_config, _simulate_summary

DATA = ROOT / "data"
RESULTS = ROOT / "results"
for d in (DATA, RESULTS):
    d.mkdir(exist_ok=True)

STRUCTURE = "S-C.2"
NOISE_PCT = 0.003
THRESH, Z0_NOM = 0.85, 0.90
N_TRAIN_CLF = 1500      # training draws for the classifier -- larger than an SBI bank is fine/cheap here
                        # (no normalizing flow, no SBC sets needed)
N_REPS, N_POST_UNUSED = 30, None  # N_REPS matches the established evaluation protocol

torch.manual_seed(20260906)
prior_wide = mixed_wide_prior()


def sample_fault_unit(theta_row, thresh=THRESH, z0_nom=Z0_NOM):
    alpha, beta_r, eta_col, xi_reb, z_A0 = theta_row
    reactor_bad = (alpha < thresh) or (beta_r < thresh)
    column_bad = (eta_col < thresh) or (xi_reb < thresh)
    feed_bad = z_A0 < thresh * z0_nom
    n_bad = int(reactor_bad) + int(column_bad) + int(feed_bad)
    if n_bad == 0:
        return "healthy"
    if n_bad >= 2:
        return "multi"
    if reactor_bad:
        return "reactor"
    if column_bad:
        return "column"
    return "feed"


## Step 1 — generate labeled training data

Reuses the exact matched-protocol simulate call (`_simulate_summary`) every other
S-C.2 bank in this project uses, so the summaries are directly comparable/reusable.
Checkpointed every 250 accepted rows (safe to interrupt and resume).


In [ ]:
CLF_BANK_PATH = DATA / f"wu2003_clf_train_n{N_TRAIN_CLF}_wideprior_stage1bfix.npz"

def generate_labeled_bank(out_path, n_rows, seed):
    if out_path.exists():
        d = np.load(out_path)
        return (np.asarray(d["thetas"], dtype=np.float32), np.asarray(d["summaries"], dtype=np.float32),
                list(d["labels"]))
    ctrl, y0 = _structure_config(STRUCTURE)
    rng = np.random.default_rng(seed)
    theta_rows, summary_rows, labels = [], [], []
    t0 = time.time()
    attempts = 0
    partial_path = out_path.with_suffix(".partial.npz")
    if partial_path.exists():
        p = np.load(partial_path, allow_pickle=True)
        theta_rows = list(np.asarray(p["thetas"], dtype=np.float32))
        summary_rows = list(np.asarray(p["summaries"], dtype=np.float32))
        labels = list(p["labels"])
        attempts = int(p["attempts"]) if "attempts" in p.files else len(theta_rows)
        print(f"resuming from partial: {len(theta_rows)}/{n_rows} rows")
    while len(theta_rows) < n_rows:
        attempts += 1
        th = prior_wide.sample((1,)).numpy()[0]
        s = _simulate_summary(th, ctrl, y0, STRUCTURE, NOISE_PCT, rng, scenario_specific_warm_start=True)
        if s is None or not np.isfinite(s).all():
            continue
        theta_rows.append(th)
        summary_rows.append(s)
        labels.append(sample_fault_unit(th))
        if len(theta_rows) % 25 == 0:
            print(f"  {len(theta_rows)}/{n_rows}, attempts={attempts} "
                  f"(accept_rate={len(theta_rows)/attempts:.3f}), elapsed={time.time()-t0:.0f}s", flush=True)
        if len(theta_rows) % 250 == 0:
            np.savez(partial_path, thetas=np.asarray(theta_rows, dtype=np.float32),
                      summaries=np.asarray(summary_rows, dtype=np.float32),
                      labels=np.asarray(labels), attempts=attempts)
    theta_arr = np.asarray(theta_rows, dtype=np.float32)
    summary_arr = np.asarray(summary_rows, dtype=np.float32)
    np.savez(out_path, thetas=theta_arr, summaries=summary_arr, labels=np.asarray(labels), attempts=attempts)
    print(f"saved {out_path}, wall_s={time.time()-t0:.0f}, accept_rate={len(theta_arr)/attempts:.3f}")
    return theta_arr, summary_arr, labels


theta_clf, summary_clf, labels_clf = generate_labeled_bank(CLF_BANK_PATH, N_TRAIN_CLF, seed=20260906001)
print("\nClass distribution in training data:")
print(pd.Series(labels_clf).value_counts())


## Step 2 — quick baseline: RandomForest

A classical-ML sanity check that needs no architecture tuning, and is often
surprisingly competitive. If this alone scores well, that's strong evidence the
information is genuinely there.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

label_to_idx = {c: i for i, c in enumerate(FAULT_UNITS)}
y_clf = np.array([label_to_idx[l] for l in labels_clf])
X_train, X_val, y_train, y_val = train_test_split(summary_clf, y_clf, test_size=0.15,
                                                    random_state=0, stratify=y_clf if len(set(y_clf)) > 1 else None)

rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=0, n_jobs=-1)
rf.fit(X_train, y_train)
print("RandomForest held-out validation (same distribution as training -- a sanity check, NOT the real test):")
print(classification_report(y_val, rf.predict(X_val), target_names=FAULT_UNITS, zero_division=0))


## Step 3 — torch MLP classifier

Matches this project's usual network-scale convention (small net, per
`feedback_network_sizing`), with class weighting for the imbalance seen in Step 1.


In [ ]:
class SmallMLP(nn.Module):
    def __init__(self, n_in, n_classes, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x):
        return self.net(x)


X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

class_counts = np.bincount(y_train, minlength=len(FAULT_UNITS)).astype(np.float32)
class_weights = torch.tensor(1.0 / np.maximum(class_counts, 1), dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * len(FAULT_UNITS)

torch.manual_seed(0)
clf_mlp = SmallMLP(n_in=N_SUMMARIES_SC2, n_classes=len(FAULT_UNITS), hidden=64)
opt = torch.optim.Adam(clf_mlp.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

best_val_loss, best_state, epochs_since_improve = float("inf"), None, 0
for epoch in range(300):
    clf_mlp.train()
    opt.zero_grad()
    loss = loss_fn(clf_mlp(X_train_t), y_train_t)
    loss.backward()
    opt.step()
    clf_mlp.eval()
    with torch.no_grad():
        val_loss = loss_fn(clf_mlp(X_val_t), y_val_t).item()
    if val_loss < best_val_loss:
        best_val_loss, best_state, epochs_since_improve = val_loss, {k: v.clone() for k, v in clf_mlp.state_dict().items()}, 0
    else:
        epochs_since_improve += 1
    if epochs_since_improve > 20:
        break
clf_mlp.load_state_dict(best_state)
print(f"MLP trained, {epoch+1} epochs, best_val_loss={best_val_loss:.4f}")
with torch.no_grad():
    val_preds = clf_mlp(X_val_t).argmax(dim=1).numpy()
print(classification_report(y_val, val_preds, target_names=FAULT_UNITS, zero_division=0))


## Step 4 — evaluate BOTH classifiers on the real 14-scenario Wu2003 catalogue

This is the test that actually matters (Steps 2-3's held-out splits are same-
distribution sanity checks only). Same protocol as every other classification script
this session (N_REPS=30 noisy replicates per scenario).


In [ ]:
def compute_classification_metrics(preds, trues, classes=FAULT_UNITS):
    n = len(classes)
    cm = np.zeros((n, n), dtype=int)
    c2i = {c: i for i, c in enumerate(classes)}
    for true, pred in zip(trues, preds):
        cm[c2i[true], c2i[pred]] += 1
    f1_per_class = {}
    for i, cls in enumerate(classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_per_class[cls] = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    macro_f1 = float(np.mean(list(f1_per_class.values())))
    accuracy = float(np.diag(cm).sum() / cm.sum()) if cm.sum() > 0 else 0.0
    return {"confusion_matrix": cm, "class_names": classes, "per_class_f1": f1_per_class,
            "macro_f1": macro_f1, "accuracy": accuracy}


y0_sb = nominal_warm_start("S-B")
rows = []
for sc in list_closed_loop():
    theta_np = np.asarray(sc.theta(), dtype=np.float32)
    true_class = sc.fault_unit()
    t_h, raw_obs = deterministic_window(sc, structure="S-B", y0=y0_sb)
    t_h_arr = np.asarray(t_h)
    rng = np.random.default_rng(20260705)
    reps = noisy_replicates(raw_obs, n_replicates=N_REPS, rng=rng, noise_pct=NOISE_PCT)
    for i in range(N_REPS):
        s = compute_summaries(reps[i], STRUCTURE, t_h_arr, theta=theta_np, rng=rng)
        rf_pred = FAULT_UNITS[rf.predict(s.reshape(1, -1))[0]]
        with torch.no_grad():
            mlp_pred = FAULT_UNITS[clf_mlp(torch.tensor(s, dtype=torch.float32).unsqueeze(0)).argmax(dim=1).item()]
        rows.append({"scenario": sc.name, "replicate": i, "true_class": true_class,
                      "rf_pred": rf_pred, "mlp_pred": mlp_pred})
    print(f"  {sc.name:22s} true={true_class}")

eval_df = pd.DataFrame(rows)
eval_df.to_csv(RESULTS / "45_direct_classifier_per_replicate.csv", index=False)

rf_metrics = compute_classification_metrics(eval_df["rf_pred"].tolist(), eval_df["true_class"].tolist())
mlp_metrics = compute_classification_metrics(eval_df["mlp_pred"].tolist(), eval_df["true_class"].tolist())

print("\n=== RandomForest on real Wu2003 catalogue ===")
print(f"macro-F1={rf_metrics['macro_f1']:.3f}  accuracy={rf_metrics['accuracy']:.3f}")
print("per-class F1:", {k: round(v, 3) for k, v in rf_metrics['per_class_f1'].items()})

print("\n=== MLP on real Wu2003 catalogue ===")
print(f"macro-F1={mlp_metrics['macro_f1']:.3f}  accuracy={mlp_metrics['accuracy']:.3f}")
print("per-class F1:", {k: round(v, 3) for k, v in mlp_metrics['per_class_f1'].items()})

np.savez(RESULTS / "45_direct_classifier_metrics.npz",
          rf_cm=rf_metrics["confusion_matrix"], mlp_cm=mlp_metrics["confusion_matrix"],
          rf_macro_f1=rf_metrics["macro_f1"], mlp_macro_f1=mlp_metrics["macro_f1"],
          rf_accuracy=rf_metrics["accuracy"], mlp_accuracy=mlp_metrics["accuracy"],
          class_names=FAULT_UNITS)


## Step 5 — context: compare against the posterior-threshold numbers

| Method | macro-F1 |
|---|---|
| full-uniform-prior SBI (nb31, pre-bugfix, historical) | 0.694 |
| narrow-lognormal SBI (S-C.2 baseline) | 0.07-0.08 |
| alpha-only mixed-prior SBI | 0.11-0.112 |
| **RandomForest (this notebook)** | *(see Step 4 output above)* |
| **MLP (this notebook)** | *(see Step 4 output above)* |

**If either classifier here scores well above 0.11** (ideally approaching or beating
0.694), that's strong evidence the summary vector's information content is fine and the
bottleneck really has been the posterior-estimation machinery (SBC/calibration/coverage),
not the underlying data — arguing for adopting a direct classifier (or a classifier-
assisted posterior-threshold hybrid) in the manuscript. **If it scores similarly poorly**,
suspect this is the SAME coverage-mismatch problem in a new guise (check the class
distribution printed in Step 1 — if `column`/`feed`/`multi` are very rare in the training
data despite `mixed_wide_prior`'s wider coverage, that's the likely cause, and the fix
would be to deliberately oversample/balance those classes when generating training data,
not to abandon the classifier idea).


## Final acceptance criteria

| # | Criterion | Threshold | Rationale |
|---|---|---|---|
| 1 | Training label class balance | every one of the 5 classes has ≥ 5 examples | prerequisite — a class with 0-4 examples can't be learned at all, regardless of classifier quality |
| 2 | Best classifier macro-F1 vs. best posterior-based result so far | ≥ 0.11 | does bypassing SBI posterior estimation show ANY promise over what's already been tried? |
| 3 | Best classifier macro-F1 (stretch) | ≥ 0.35 | roughly half the historical full-uniform-prior ceiling (0.694) |


In [ ]:
class_counts_check = pd.Series(labels_clf).value_counts()
min_class_count = min(class_counts_check.get(u, 0) for u in FAULT_UNITS)
best_f1 = max(rf_metrics["macro_f1"], mlp_metrics["macro_f1"])
best_name = "RandomForest" if rf_metrics["macro_f1"] >= mlp_metrics["macro_f1"] else "MLP"

criteria = [
    {"criterion": "1. Every class has >= 5 training examples", "value": min_class_count,
     "threshold": 5, "pass": min_class_count >= 5},
    {"criterion": "2. Best classifier macro-F1 >= 0.11", "value": best_f1,
     "threshold": 0.11, "pass": best_f1 >= 0.11},
    {"criterion": "3. Best classifier macro-F1 >= 0.35 (stretch)", "value": best_f1,
     "threshold": 0.35, "pass": best_f1 >= 0.35},
]

print("=" * 78)
print(f"{'Criterion':<45s} {'Value':>10s} {'Threshold':>10s} {'Result':>8s}")
print("-" * 78)
for c in criteria:
    print(f"{c['criterion']:<45s} {c['value']:>10.3g} {c['threshold']:>10.2f} {'PASS' if c['pass'] else 'FAIL':>8s}")
print("=" * 78)
print(f"\nBest classifier: {best_name} (macro-F1={best_f1:.3f})")

if not criteria[0]["pass"]:
    print("\nVERDICT: INVALID -- at least one class is too rare in the training data to "
          "have been learnable. Check the class distribution printed in Step 1; the fix "
          "is to oversample/balance rare classes when generating training data, not to "
          "abandon the classifier idea.")
elif criteria[2]["pass"]:
    print("\nVERDICT: PASS -- the summary vector's information content is fine; the "
          "bottleneck has been posterior-estimation machinery (SBC/coverage), not the "
          "underlying data. Worth adopting a direct/hybrid classifier in the manuscript.")
elif criteria[1]["pass"]:
    print("\nVERDICT: PARTIAL -- beats the best posterior-based result so far but isn't "
          "yet a strong result on its own.")
else:
    print("\nVERDICT: FAIL -- this recurs the same coverage-mismatch problem in a new "
          "guise (or the summary vector itself doesn't cleanly separate classes). Check "
          "class balance first before concluding the information isn't there.")
